# 04 — Novel Composition Exploration

Experiment with substituting structurally similar boxes between algorithms.
If motif M appears in both algorithm A and B, what happens when we swap them?

In [1]:
import copy

import pandas as pd
import pyzx as zx
from qiskit import QuantumCircuit

from zx_motifs.pipeline.composer import (
    compose_sequential,
    make_box_from_circuit,
    simplify_box,
    BoundaryDestroyedError,
)

## Build Sub-Algorithm Boxes

In [2]:
n = 3

# Superposition
qc_sup = QuantumCircuit(n)
qc_sup.h(range(n))
sup_box = make_box_from_circuit("superposition", qc_sup, "superposition")

# Grover oracle
qc_oracle = QuantumCircuit(n)
qc_oracle.x(range(n))
qc_oracle.h(n - 1)
qc_oracle.mcx(list(range(n - 1)), n - 1)
qc_oracle.h(n - 1)
qc_oracle.x(range(n))
oracle_box = make_box_from_circuit("grover_oracle", qc_oracle, "oracle")

# Grover diffusion
qc_diff = QuantumCircuit(n)
qc_diff.h(range(n))
qc_diff.x(range(n))
qc_diff.h(n - 1)
qc_diff.mcx(list(range(n - 1)), n - 1)
qc_diff.h(n - 1)
qc_diff.x(range(n))
qc_diff.h(range(n))
diffusion_box = make_box_from_circuit("diffusion", qc_diff, "diffusion")

# QAOA mixer
qc_mixer = QuantumCircuit(n)
for i in range(n):
    qc_mixer.rx(0.5, i)
mixer_box = make_box_from_circuit("qaoa_mixer", qc_mixer, "mixer")

print("Boxes built:")
for box in [sup_box, oracle_box, diffusion_box, mixer_box]:
    print(f"  {box.name}: {box.n_inputs}→{box.n_outputs}, "
          f"{box.graph.num_vertices()}V {box.graph.num_edges()}E")

Boxes built:
  superposition: 3→3, 9V 6E
  grover_oracle: 3→3, 35V 38E
  diffusion: 3→3, 41V 44E
  qaoa_mixer: 3→3, 9V 6E


## Mixer Substitution Experiment

What happens when you replace Grover's diffusion with a QAOA mixer?

In [3]:
# Standard Grover: sup -> oracle -> diffusion
standard = compose_sequential(sup_box, oracle_box)
standard = compose_sequential(standard, diffusion_box)

# Hybrid: sup -> oracle -> QAOA mixer
hybrid = compose_sequential(sup_box, oracle_box)
hybrid = compose_sequential(hybrid, mixer_box)

# Simplify both
standard_simp = simplify_box(standard, level="interior_clifford")
hybrid_simp = simplify_box(hybrid, level="interior_clifford")

print(f"Standard Grover (simplified):      V={standard_simp.graph.num_vertices()} "
      f"E={standard_simp.graph.num_edges()}")
print(f"Hybrid Oracle+Mixer (simplified):  V={hybrid_simp.graph.num_vertices()} "
      f"E={hybrid_simp.graph.num_edges()}")

# Are they the same unitary?
same = zx.compare_tensors(standard_simp.graph, hybrid_simp.graph)
print(f"\nSame unitary? {same}")
print("(Expected: False — the QAOA mixer is a different operation)")

Standard Grover (simplified):      V=34 E=43
Hybrid Oracle+Mixer (simplified):  V=23 E=26

Same unitary? False
(Expected: False — the QAOA mixer is a different operation)


## Systematic Exploration Sweep

For all pairs of boxes with matching boundaries, compare their
ZX complexity after simplification.

In [4]:
import numpy as np

# Build a QFT stage box at the same width for comparison
qc_qft_stage = QuantumCircuit(n)
qc_qft_stage.h(0)
for j in range(1, n):
    qc_qft_stage.cp(np.pi / (2 ** j), j, 0)
qft_stage_box = make_box_from_circuit("qft_stage", qc_qft_stage, "qft_butterfly")

box_library = [sup_box, oracle_box, diffusion_box, mixer_box, qft_stage_box]

results = []
for i, box_a in enumerate(box_library):
    for j, box_b in enumerate(box_library):
        if i == j:
            continue
        if box_a.n_inputs != box_b.n_inputs or box_a.n_outputs != box_b.n_outputs:
            continue

        g_a = copy.deepcopy(box_a.graph)
        g_b = copy.deepcopy(box_b.graph)

        zx.simplify.full_reduce(g_a)
        zx.simplify.full_reduce(g_b)

        results.append({
            "box_a": box_a.name,
            "box_b": box_b.name,
            "a_vertices": g_a.num_vertices(),
            "b_vertices": g_b.num_vertices(),
            "a_edges": g_a.num_edges(),
            "b_edges": g_b.num_edges(),
            "vertex_diff": g_a.num_vertices() - g_b.num_vertices(),
            "edge_diff": g_a.num_edges() - g_b.num_edges(),
            "same_unitary": zx.compare_tensors(g_a, g_b),
        })

sweep_df = pd.DataFrame(results)
print(sweep_df.to_string())

            box_a          box_b  a_vertices  b_vertices  a_edges  b_edges  vertex_diff  edge_diff  same_unitary
0   superposition  grover_oracle           6          23        3       31          -17        -28         False
1   superposition      diffusion           6          23        3       31          -17        -28         False
2   superposition     qaoa_mixer           6           9        3        6           -3         -3         False
3   superposition      qft_stage           6          13        3       12           -7         -9         False
4   grover_oracle  superposition          23           6       31        3           17         28         False
5   grover_oracle      diffusion          23          23       31       31            0          0         False
6   grover_oracle     qaoa_mixer          23           9       31        6           14         25         False
7   grover_oracle      qft_stage          23          13       31       12           10         

## Interpretation Guide

- **same_unitary = True, different complexity**: You found an optimization — one box is simpler.
- **same_unitary = False, similar complexity**: Structurally similar but semantically different.
- **vertex_diff > 0**: box_b is more compact after full reduction.

Pairs where `same_unitary = True` and `vertex_diff != 0` are the most interesting —
they suggest one decomposition is inherently simpler in the ZX representation.